A

In [20]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m02")

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated


esquema de pedidos

In [21]:
from pyspark.sql.types import StructType, StructField, StringType

orders_raw_schema = StructType([
    StructField("OrderId", StringType(), True),
    StructField("CustomerId", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Channel", StringType(), True),
])
orders = (
    spark.read.option("header", True)
    .schema(orders_raw_schema)
    .csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
)
orders.printSchema()
print(orders.count())


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts_raw: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)

800


In [28]:
from pyspark.sql.functions import col, coalesce, to_timestamp

orders = orders.withColumn(
    "order_ts",
    coalesce(
        to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
    ),
)
print("nulos de fecha", orders.where(col("order_ts").isNull()).count())
orders.printSchema()

nulos de fecha 0
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts_raw: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)



In [23]:
print("nulos de fecha", orders.where(col("order_ts_raw").isNull()).count())
orders.show()

nulos de fecha 0
+--------+-----------+-------------------+---------+-----------+-------------------+
|order_id|customer_id|       order_ts_raw|   status|    channel|           order_ts|
+--------+-----------+-------------------+---------+-----------+-------------------+
|  O00001|       NULL|2024-04-16 20:00:00|     paid|        app|2024-04-16 20:00:00|
|  O00002|       NULL|2024-12-04 02:00:00|     paid|      store|2024-12-04 02:00:00|
|  O00003|       NULL|2024-03-29 17:00:00|     paid|        WEB|2024-03-29 17:00:00|
|  O00004|       NULL|2024-01-02 13:00:00|cancelled|        App|2024-01-02 13:00:00|
|  O00005|       NULL|2024-05-29 01:00:00|cancelled|marketplace|2024-05-29 01:00:00|
|  O00006|       NULL|2024-12-27 09:00:00| refunded|        web|2024-12-27 09:00:00|
|  O00007|       NULL|2024-02-06 21:00:00| refunded|        app|2024-02-06 21:00:00|
|  O00008|       NULL|2024-11-16 18:00:00| refunded|        web|2024-11-16 18:00:00|
|  O00009|       NULL|2024-08-05 03:00:00|     p

cargo qty

In [24]:
from pyspark.sql.types import IntegerType, DecimalType

items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
items.printSchema()
items.select("unit_price").limit(3).show()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount: decimal(5,2) (nullable = true)

+----------+
|unit_price|
+----------+
|    114.41|
|     90.86|
|     16.79|
+----------+



cargo origen tipo json

In [25]:
from pyspark.sql.types import TimestampType

events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("session_id", StringType(), True),
    StructField("page", StringType(), True),
    StructField("product_id", StringType(), True),
])
events = spark.read.schema(events_schema).json(str(RAW / "events.jsonl"))
events.printSchema()
print(events.count())


root
 |-- event_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)

2500


In [26]:
events.show()

+--------+-----------+-----------+-------------------+----------+---------+----------+
|event_id|customer_id| event_type|                 ts|session_id|     page|product_id|
+--------+-----------+-----------+-------------------+----------+---------+----------+
| E000001|       NULL|  page_view|2024-08-11 23:46:00|     S0806| /catalog|      P038|
| E000002|       NULL|  page_view|2024-07-05 11:43:00|     S0636|/checkout|      P058|
| E000003|       NULL|  page_view|2024-07-24 22:03:00|     S0055|    /cart|      P009|
| E000004|       NULL|  page_view|2024-05-26 16:57:00|     S0474|/checkout|      P030|
| E000005|       NULL|  page_view|2024-11-01 13:01:00|     S0057| /product|      P054|
| E000006|       NULL|  page_view|2024-12-25 01:18:00|     S0342|/checkout|      P007|
| E000007|       NULL|add_to_cart|2024-10-26 06:13:00|     S0494| /product|      P056|
| E000008|       NULL|   purchase|2024-01-23 03:50:00|     S0599|/checkout|      P048|
| E000009|       NULL|   purchase|2024-09-1

In [27]:
products = (
    spark.read.option("multiLine", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
)
products.printSchema()

root
 |-- category: string (nullable = true)
 |-- list_price: decimal(10,2) (nullable = true)
 |-- name: string (nullable = true)
 |-- product_id: string (nullable = true)

